# PROJECT : 2

## Movie Recommendation Engine

### Phase:1 Load & Merge Data

In [1]:
import pandas as pd

movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

# Merge datasets
movies = movies.merge(credits, on="title")

print("Shape:", movies.shape)

print("\nColumns:")
print(movies.columns)

print("\nMissing Values:")
print(movies.isnull().sum())

print("\nVote Average Statistics:")
print(movies["vote_average"].describe())

Shape: (4809, 23)

Columns:
Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count', 'movie_id', 'cast', 'crew'],
      dtype='object')

Missing Values:
budget                     0
genres                     0
homepage                3096
id                         0
keywords                   0
original_language          0
original_title             0
overview                   3
popularity                 0
production_companies       0
production_countries       0
release_date               1
revenue                    0
runtime                    2
spoken_languages           0
status                     0
tagline                  844
title                      0
vote_average               0
vote_count                 0
movie_id

### Phase:2 Content Feature Engineering

In [2]:
# Required Library
import ast

# Genres
def get_genres(text):
    genres = []
    for item in ast.literal_eval(text):
        genres.append(item['name'].replace(" ",""))
    return genres[:3]

# Keywords
def get_keywords(text):
    keywords = []
    for item in ast.literal_eval(text):
        keywords.append(item['name'].replace(" ",""))
    return keywords[:5]

# Cast
def get_cast(text):
    cast = []
    for item in ast.literal_eval(text):
        cast.append(item['name'].replace(" ",""))
    return cast[:3]

# Director
def get_director(text):
    crew = ast.literal_eval(text)

    for member in crew:
        if member['job'] == 'Director':
            return member['name'].replace(" ","")
    return ""

movies['genres'] = movies['genres'].apply(get_genres)
movies['keywords'] = movies['keywords'].apply(get_keywords)
movies['cast'] = movies['cast'].apply(get_cast)
movies['director'] = movies['crew'].apply(get_director)

movies['content_soup'] = (
    movies['genres'].apply(lambda x: " ".join(x))
    + " " +
    movies['keywords'].apply(lambda x: " ".join(x))
    + " " +
    movies['cast'].apply(lambda x: " ".join(x))
    + " " +
    movies['director']
)

print(movies[['title','content_soup']].head())

                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   
3                     The Dark Knight Rises   
4                               John Carter   

                                        content_soup  
0  Action Adventure Fantasy cultureclash future s...  
1  Adventure Fantasy Action ocean drugabuse exoti...  
2  Action Adventure Crime spy basedonnovel secret...  
3  Action Crime Drama dccomics crimefighter terro...  
4  Action Adventure ScienceFiction basedonnovel m...  


### Phase:3 TF-IDF Similarity Matrix

In [3]:
# Required Libraries
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(movies['content_soup'])

similarity = cosine_similarity(tfidf_matrix)

print("Similarity Matrix Shape:")
print(similarity.shape)

memory_mb = similarity.nbytes / (1024 * 1024)

print("Memory Usage:", round(memory_mb,2),"MB")

Similarity Matrix Shape:
(4809, 4809)
Memory Usage: 176.44 MB


### Phase:4 Content-Based Recommender

In [4]:
# Required Library
from difflib import get_close_matches

def get_recommendations(title, n=10):

    titles = movies['title'].tolist()

    match = get_close_matches(title, titles, n=1)

    if len(match)==0:
        return "Movie Not Found"

    title = match[0]

    idx = movies[movies['title']==title].index[0]

    scores = list(enumerate(similarity[idx]))

    scores = sorted(scores,key=lambda x:x[1],reverse=True)

    scores = scores[1:n+1]

    result = []

    for i,score in scores:

        result.append({
            "Title": movies.iloc[i]['title'],
            "Genres": movies.iloc[i]['genres'],
            "Rating": movies.iloc[i]['vote_average'],
            "Similarity": round(score,3)
        })

    return pd.DataFrame(result)

# Testing
print(get_recommendations("The Dark Knight"))
print(get_recommendations("Inception"))
print(get_recommendations("Toy Story"))

                      Title                        Genres  Rating  Similarity
0             Batman Begins        [Action, Crime, Drama]     7.5       0.548
1     The Dark Knight Rises        [Action, Crime, Drama]     7.6       0.540
2              The Prestige    [Drama, Mystery, Thriller]     8.0       0.236
3            Batman Returns             [Action, Fantasy]     6.6       0.210
4            Batman & Robin      [Action, Crime, Fantasy]     4.2       0.208
5  Amidst the Devil's Wings        [Drama, Action, Crime]     0.0       0.202
6                    Batman             [Fantasy, Action]     7.0       0.196
7                  Superman  [Action, Adventure, Fantasy]     6.9       0.194
8                Kick-Ass 2    [Action, Adventure, Crime]     6.3       0.191
9                  Kick-Ass               [Action, Crime]     7.1       0.190
                        Title                              Genres  Rating  \
0                     Don Jon            [Romance, Comedy, Drama]

### Phase:5 Collaborative Filtering

In [5]:
# Required Library
from sklearn.decomposition import TruncatedSVD

print("""
Collaborative Filtering Skipped

TMDB dataset contains movie metadata only.

No user ratings available.

Content-based filtering solves cold-start problem.
""")


Collaborative Filtering Skipped

TMDB dataset contains movie metadata only.

No user ratings available.

Content-based filtering solves cold-start problem.



### Phase:6 Evaluation (Precision@10)

In [6]:
# Required Libraries
import numpy as np
import random

def precision_at_k(title,k=10):

    try:
        idx = movies[movies['title']==title].index[0]

        original_genres = set(movies.iloc[idx]['genres'])

        recs = get_recommendations(title,k)

        relevant = 0

        for rec_title in recs['Title']:

            rec_idx = movies[movies['title']==rec_title].index[0]

            rec_genres = set(movies.iloc[rec_idx]['genres'])

            overlap = len(
                original_genres.intersection(rec_genres)
            )

            if overlap >= 2:
                relevant += 1

        return relevant/k

    except:
        return 0

sample_movies = random.sample(
    movies['title'].tolist(),
    100
)

scores = []

for movie in sample_movies:
    scores.append(
        precision_at_k(movie)
    )

avg_precision = np.mean(scores)

print("Average Precision@10:", round(avg_precision,3))

Average Precision@10: 0.337


### Phase:7 Hybrid Recommender

In [7]:
class HybridRecommender:

    def recommend(self,title,n=10):

        print("Using Content-Based Recommendation")

        return get_recommendations(title,n)

hybrid = HybridRecommender()

print(hybrid.recommend("Inception"))

Using Content-Based Recommendation
                        Title                              Genres  Rating  \
0                     Don Jon            [Romance, Comedy, Drama]     5.9   
1                      Looper  [Action, Thriller, ScienceFiction]     6.6   
2                Premium Rush           [Crime, Action, Thriller]     6.2   
3        (500) Days of Summer            [Comedy, Drama, Romance]     7.2   
4                   The Juror                   [Drama, Thriller]     5.5   
5                    The Walk        [Adventure, Drama, Thriller]     6.9   
6                   Stop-Loss                        [Drama, War]     6.1   
7                       50/50                     [Comedy, Drama]     7.0   
8  10 Things I Hate About You            [Comedy, Romance, Drama]     7.3   
9             Treasure Planet      [Adventure, Animation, Family]     7.2   

   Similarity  
0       0.310  
1       0.250  
2       0.201  
3       0.194  
4       0.193  
5       0.190  
6    

### Phase:8 Production Recommendation API

In [8]:
# Required Library
import time

class RecommendationAPI:

    def recommend(self,title):

        return get_recommendations(title)

    def explain(self,title,recommendation):

        return (
            f"{recommendation} was recommended "
            f"because it shares genres, cast, "
            f"director and keywords with {title}"
        )

    def batch_recommend(self,titles):

        results = {}

        for movie in titles:

            results[movie] = self.recommend(movie)

        return results

api = RecommendationAPI()

start = time.time()

api.recommend("Inception")

end = time.time()

print(
    "Recommendation Time:",
    round((end-start)*1000,2),
    "ms"
)

Recommendation Time: 37.03 ms
